# 🎬 Movie Recommendation System - Complete Analysis

This notebook builds a movie recommendation system using:
1. **Collaborative Filtering** - Find similar users
2. **Content-Based Filtering** - Find similar movies by genre
3. **Data Analysis** - Understand patterns in the data

---

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Import our custom recommender
import sys
sys.path.append('../src')
from recommender import MovieRecommender

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")

## Step 2: Load and Explore Data

First, we need to get the MovieLens dataset. You can download from:
- https://grouplens.org/datasets/movielens/latest-small/

Or run the code below to download automatically:

In [ ]:
# Download MovieLens dataset
import urllib.request
import zipfile
import os

# Create data directory if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Download if not exists
if not os.path.exists('../data/ratings.csv'):
    print("📥 Downloading MovieLens dataset...")
    url = 'http://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
    urllib.request.urlretrieve(url, '../data/movielens.zip')
    
    # Extract
    with zipfile.ZipFile('../data/movielens.zip', 'r') as zip_ref:
        zip_ref.extractall('../data')
    
    # Move files
    import shutil
    shutil.move('../data/ml-latest-small/ratings.csv', '../data/ratings.csv')
    shutil.move('../data/ml-latest-small/movies.csv', '../data/movies.csv')
    print("✅ Dataset downloaded!")
else:
    print("✅ Dataset already exists!")

In [ ]:
# Load the data
ratings_df = pd.read_csv('../data/ratings.csv')
movies_df = pd.read_csv('../data/movies.csv')

print("📊 RATINGS DATA:")
print(f"Shape: {ratings_df.shape}")
print(f"Columns: {ratings_df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(ratings_df.head())
print(f"\nData types:\n{ratings_df.dtypes}")

print("\n" + "="*50)
print("🎬 MOVIES DATA:")
print(f"Shape: {movies_df.shape}")
print(f"Columns: {movies_df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(movies_df.head())

## Step 3: Exploratory Data Analysis (EDA)

**What we're doing:**
- Understanding the distribution of ratings
- Finding most-rated movies
- Understanding user behavior


In [ ]:
# Basic statistics
print("📈 RATINGS STATISTICS:")
print(ratings_df['rating'].describe())

print(f"\n📊 Total Users: {ratings_df['userId'].nunique()}")
print(f"🎬 Total Movies: {ratings_df['movieId'].nunique()}")
print(f"⭐ Total Ratings: {len(ratings_df)}")
print(f"📉 Sparsity: {(1 - len(ratings_df) / (ratings_df['userId'].nunique() * ratings_df['movieId'].nunique())) * 100:.2f}%")

In [ ]:
# Visualization 1: Rating Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ratings_df['rating'], bins=20, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Movie Ratings')
axes[0].grid(axis='y', alpha=0.3)

# Box plot
axes[1].boxplot(ratings_df['rating'])
axes[1].set_ylabel('Rating')
axes[1].set_title('Box Plot of Ratings')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/01_rating_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: 01_rating_distribution.png")

In [ ]:
# Top 10 Most Rated Movies
top_rated_counts = ratings_df['movieId'].value_counts().head(10)
top_rated_movies = movies_df[movies_df['movieId'].isin(top_rated_counts.index)]
top_rated_movies = top_rated_movies.merge(
    ratings_df.groupby('movieId')['rating'].mean().rename('avg_rating'),
    left_on='movieId',
    right_index=True
)
top_rated_movies = top_rated_movies.merge(
    top_rated_counts.rename('num_ratings'),
    left_on='movieId',
    right_index=True
)

print("🏆 TOP 10 MOST RATED MOVIES:")
print(top_rated_movies[['title', 'avg_rating', 'num_ratings']].to_string())

In [ ]:
# Visualization 2: Top Rated Movies
fig, ax = plt.subplots(figsize=(12, 6))

movies_plot = top_rated_movies.head(10).sort_values('num_ratings')
ax.barh(range(len(movies_plot)), movies_plot['num_ratings'], color='steelblue')
ax.set_yticks(range(len(movies_plot)))
ax.set_yticklabels([t[:40] for t in movies_plot['title']], fontsize=10)
ax.set_xlabel('Number of Ratings')
ax.set_title('Top 10 Most Rated Movies')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/02_top_rated_movies.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: 02_top_rated_movies.png")

In [ ]:
# User behavior analysis
user_stats = ratings_df.groupby('userId').agg({
    'rating': ['count', 'mean', 'std']
}).reset_index()
user_stats.columns = ['userId', 'num_ratings', 'mean_rating', 'std_rating']

print("👥 USER BEHAVIOR STATISTICS:")
print(user_stats[['num_ratings', 'mean_rating', 'std_rating']].describe())

# Visualization 3: User Activity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_stats['num_ratings'], bins=50, color='coral', edgecolor='black')
axes[0].set_xlabel('Number of Ratings per User')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of User Ratings Count')
axes[0].set_yscale('log')
axes[0].grid(axis='y', alpha=0.3)

axes[1].scatter(user_stats['num_ratings'], user_stats['mean_rating'], alpha=0.5, s=20)
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Average Rating')
axes[1].set_title('User Activity vs Average Rating')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/03_user_behavior.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: 03_user_behavior.png")

## Step 4: Prepare Data for Recommendation

**What we're doing:**
- Creating the User-Item matrix
- Preparing data for collaborative filtering

In [ ]:
# Initialize recommender
recommender = MovieRecommender(ratings_df, movies_df)

# Create user-item matrix
user_item_matrix = recommender.create_user_item_matrix()

print(f"✅ User-Item Matrix created!")
print(f"Shape: {user_item_matrix.shape}")
print(f"\nFirst 5x5 of the matrix:")
print(user_item_matrix.iloc[:5, :5])
print(f"\nMatrix sparsity: {(user_item_matrix == 0).sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]) * 100:.2f}%")

## Step 5: Build Recommendation Models

### Model 1: Collaborative Filtering (User-Based)
**How it works:**
1. Find users similar to our target user
2. See what movies they rated highly
3. Recommend those movies


In [ ]:
# Test Collaborative Filtering
# Let's get recommendations for user 1
user_id = 1

print(f"🎯 Getting recommendations for User {user_id}...\n")

# Get collaborative filtering recommendations
cf_recommendations = recommender.collaborative_filtering(user_id, n_recommendations=10)

print("🤝 COLLABORATIVE FILTERING RECOMMENDATIONS:")
print(cf_recommendations.to_string())

print(f"\n✅ Top recommended movie: {cf_recommendations.iloc[0]['title']}")

### Model 2: Content-Based Filtering
**How it works:**
1. Find a movie the user liked
2. Find other movies with similar genres
3. Recommend those similar movies

In [ ]:
# Let's take a movie that user 1 rated highly
user_ratings = ratings_df[ratings_df['userId'] == user_id].sort_values('rating', ascending=False)

if len(user_ratings) > 0:
    top_rated_by_user = user_ratings.iloc[0]
    movie_id = int(top_rated_by_user['movieId'])
    movie_title = movies_df[movies_df['movieId'] == movie_id]['title'].values[0]
    rating = top_rated_by_user['rating']
    
    print(f"🎬 User {user_id} rated '{movie_title}' with {rating} stars\n")
    
    # Get content-based recommendations
    cb_recommendations = recommender.content_based_filtering(movie_id, n_recommendations=10)
    
    print("📚 CONTENT-BASED FILTERING RECOMMENDATIONS:")
    print("(Movies similar by genre)\n")
    print(cb_recommendations.to_string())
    

## Step 6: Evaluation Metrics

**How we measure if recommendations are good:**

In [ ]:
# Calculate RMSE for a simple baseline
# Predict ratings as the average rating for each movie

movie_avg_ratings = ratings_df.groupby('movieId')['rating'].mean()

# Calculate predictions
ratings_df_copy = ratings_df.copy()
ratings_df_copy['predicted_rating'] = ratings_df_copy['movieId'].map(movie_avg_ratings)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(ratings_df_copy['rating'], ratings_df_copy['predicted_rating']))

print(f"📊 MODEL EVALUATION METRICS:")
print(f"\nBaseline RMSE (predicting movie average): {rmse:.4f}")
print(f"\nExplanation:")
print(f"- RMSE measures prediction error (lower is better)")
print(f"- Our baseline RMSE of {rmse:.4f} means predictions are off by ~{rmse:.2f} stars on average")
print(f"- Our collaborative filtering should beat this!")

## Step 7: Multiple User Examples

Let's show recommendations for multiple users:

In [ ]:
# Get recommendations for 3 different users
test_users = [1, 5, 10]

for test_user in test_users:
    print(f"\n{'='*60}")
    print(f"👤 RECOMMENDATIONS FOR USER {test_user}")
    print(f"{'='*60}")
    
    # Get user's rating history
    user_history = ratings_df[ratings_df['userId'] == test_user].merge(
        movies_df, on='movieId'
    ).sort_values('rating', ascending=False).head(3)
    
    print(f"\n⭐ User's top-rated movies:")
    for idx, (_, row) in enumerate(user_history.iterrows(), 1):
        print(f"  {idx}. {row['title']} - Rating: {row['rating']}")
    
    # Get recommendations
    recs = recommender.collaborative_filtering(test_user, n_recommendations=5)
    
    print(f"\n🎁 Recommended movies:")
    for idx, (_, rec) in enumerate(recs.iterrows(), 1):
        print(f"  {idx}. {rec['title']}")
        print(f"     Genre: {rec['genres']}")
        print(f"     Score: {rec['recommendation_score']:.2f}")

## Step 8: Visualization - Recommendation Comparison


In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Movie Recommendation System - Analysis Summary', fontsize=16, fontweight='bold')

# Plot 1: Rating distribution
axes[0, 0].hist(ratings_df['rating'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Rating Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Top movies by average rating
top_avg = ratings_df.groupby('movieId')['rating'].mean().nlargest(10)
top_movies_avg = movies_df[movies_df['movieId'].isin(top_avg.index)]
top_movies_avg = top_movies_avg.merge(
    top_avg.rename('avg_rating'),
    left_on='movieId',
    right_index=True
).sort_values('avg_rating')

axes[0, 1].barh(range(len(top_movies_avg)), top_movies_avg['avg_rating'], color='coral')
axes[0, 1].set_yticks(range(len(top_movies_avg)))
axes[0, 1].set_yticklabels([t[:30] for t in top_movies_avg['title']], fontsize=9)
axes[0, 1].set_title('Top 10 Movies by Average Rating', fontweight='bold')
axes[0, 1].set_xlabel('Average Rating')
axes[0, 1].grid(axis='x', alpha=0.3)

# Plot 3: User activity distribution
axes[1, 0].hist(user_stats['num_ratings'], bins=50, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('User Activity Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Number of Ratings per User')
axes[1, 0].set_ylabel('Number of Users')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 4: Statistics summary
axes[1, 1].axis('off')
summary_text = f"""
📊 DATASET STATISTICS:

👥 Total Users: {ratings_df['userId'].nunique():,}
🎬 Total Movies: {ratings_df['movieId'].nunique():,}
⭐ Total Ratings: {len(ratings_df):,}

📈 Rating Statistics:
   • Mean: {ratings_df['rating'].mean():.2f}
   • Std Dev: {ratings_df['rating'].std():.2f}
   • Min: {ratings_df['rating'].min():.1f}
   • Max: {ratings_df['rating'].max():.1f}

👤 User Statistics:
   • Avg ratings per user: {user_stats['num_ratings'].mean():.1f}
   • Max ratings from user: {user_stats['num_ratings'].max()}
   • Min ratings from user: {user_stats['num_ratings'].min()}

📉 Data Sparsity: {(1 - len(ratings_df) / (ratings_df['userId'].nunique() * ratings_df['movieId'].nunique())) * 100:.2f}%
"""

axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center',
                fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('../results/04_complete_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: 04_complete_analysis.png")

## Step 9: Summary & Key Takeaways


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║        🎬 MOVIE RECOMMENDATION SYSTEM - PROJECT SUMMARY 🎬        ║
╚════════════════════════════════════════════════════════════════╝

✅ WHAT WE BUILT:
   1. Data Exploration & Analysis (EDA)
   2. Collaborative Filtering (User-Based Recommendations)
   3. Content-Based Filtering (Genre-Based Recommendations)
   4. Evaluation Metrics (RMSE)
   5. Visualizations & Insights

📊 KEY ALGORITHMS:

   🤝 Collaborative Filtering:
      • Finds similar users based on rating patterns
      • Recommends movies liked by similar users
      • Formula: Similarity = Cosine(User1, User2)
      • Best for: Discovering new movies

   📚 Content-Based Filtering:
      • Finds movies similar by genre
      • Recommends movies with similar content
      • Formula: Similarity = Shared Genres / Total Genres
      • Best for: Finding related content

📈 RESULTS:
   • Successfully generates top-N personalized recommendations
   • User-item matrix: {user_item_matrix.shape}
   • Sparsity: {(1 - len(ratings_df) / (ratings_df['userId'].nunique() * ratings_df['movieId'].nunique())) * 100:.2f}%

🎯 IMPROVEMENTS FOR PRODUCTION:
   1. Matrix Factorization (SVD) for better accuracy
   2. Deep Learning (Neural Collaborative Filtering)
   3. Hybrid approach (combining both methods)
   4. Cold Start Problem handling
   5. Real-time updates with new ratings

""")